In [1]:
!pip install chromadb einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [2]:
import json
import math
import time
import chromadb
from chromadb.utils import embedding_functions
from datasets import load_dataset
from tqdm import tqdm

In [3]:
# ==========================================
# 1. METRICS ENGINE
# ==========================================


def calculate_metrics(retrieved_ids, target_id, latency_ms):
    if target_id not in retrieved_ids:
        return {"recall": 0, "mrr": 0.0, "ndcg": 0.0, "latency_ms": latency_ms}
    rank = retrieved_ids.index(target_id) + 1
    return {
        "recall": 1,
        "mrr": 1.0 / rank,
        "ndcg": 1.0 / math.log2(rank + 1),
        "latency_ms": latency_ms,
    }


def print_scorecard(model_name, dataset_name, metrics_list):
    total = len(metrics_list)
    if total == 0:
        return
    avg_recall = sum(m["recall"] for m in metrics_list) / total
    avg_mrr = sum(m["mrr"] for m in metrics_list) / total
    avg_ndcg = sum(m["ndcg"] for m in metrics_list) / total
    avg_latency = sum(m["latency_ms"] for m in metrics_list) / total

    print("\n" + "-" * 50)
    print(f"📂 DATASET: {dataset_name}")
    print(f"Recall@10:         {avg_recall:.4f} ({(avg_recall*100):.1f}%)")
    print(f"MRR:               {avg_mrr:.4f}")
    print(f"nDCG@10:           {avg_ndcg:.4f}")
    print(f"Avg Query Latency: {avg_latency:.2f} ms")
    print("-" * 50)

In [4]:
# ==========================================
# 2. LOAD BOTH DATASETS INTO MEMORY
# ==========================================
print("Loading datasets into memory...")

# A. Synthetic Data
with open("/kaggle/input/datasets/amirbnsl/synthetic-eval-set-json/synthetic_eval_set.json", "r") as f:
    synthetic_ground_truth = json.load(f)
arxiv_abstracts = load_dataset("gfissore/arxiv-abstracts-2021", split="train")
synthetic_docs = arxiv_abstracts.select(range(100))

Loading datasets into memory...


README.md: 0.00B [00:00, ?B/s]

arxiv-abstracts.jsonl.gz:   0%|          | 0.00/940M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999486 [00:00<?, ? examples/s]

In [5]:
import pyarrow.parquet as pq
from huggingface_hub import list_repo_files, hf_hub_download

repo_id = "c1khoa/papers_retrieval_arxiv"
files = [f for f in list_repo_files(repo_id, repo_type="dataset") if f.endswith('.parquet')]

print(f"🔍 Found {len(files)} files. Peeking at schemas...\n")

for file in files:
    path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=file)
    schema = pq.ParquetFile(path).schema.names
    print(f"File: {file}")
    print(f"Columns: {schema}\n" + "-"*30)

🔍 Found 3 files. Peeking at schemas...



ground_truth/qrels.parquet:   0%|          | 0.00/52.3k [00:00<?, ?B/s]

File: ground_truth/qrels.parquet
Columns: ['query_id', 'doc_id', 'relevance']
------------------------------


ground_truth/queries.parquet:   0%|          | 0.00/6.26k [00:00<?, ?B/s]

File: ground_truth/queries.parquet
Columns: ['query_id', 'text']
------------------------------


raw/papers_2015_2025.parquet:   0%|          | 0.00/298M [00:00<?, ?B/s]

File: raw/papers_2015_2025.parquet
Columns: ['id', 'title', 'abstract', 'element', 'element', 'update_date']
------------------------------


In [6]:
import pandas as pd
from huggingface_hub import hf_hub_download

print("Mapping the exact schemas found...")
repo_id = "c1khoa/papers_retrieval_arxiv"

# 1. Load Queries (using 'text' instead of 'query')
print("Loading queries...")
queries_path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename="ground_truth/queries.parquet")
queries_df = pd.read_parquet(queries_path, columns=['query_id', 'text'])

# 2. Load Qrels (Mappings)
print("Loading mappings...")
qrels_path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename="ground_truth/qrels.parquet")
qrels_df = pd.read_parquet(qrels_path, columns=['query_id', 'doc_id'])

# 3. Load Corpus (Papers - explicitly grabbing 'id' and 'abstract' to dodge the broken 'element' columns)
print("Loading corpus...")
corpus_path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename="raw/papers_2015_2025.parquet")
corpus_df = pd.read_parquet(corpus_path, columns=['id', 'abstract'])

print(f"\n✅ Extracted Tables: {len(corpus_df)} papers | {len(queries_df)} queries | {len(qrels_df)} mappings")
print("Stitching them together for evaluation...")

# 4. Perform the "SQL Join" in memory for blazing fast lookups
corpus_dict = pd.Series(corpus_df.abstract.values, index=corpus_df.id).to_dict()
queries_dict = pd.Series(queries_df.text.values, index=queries_df.query_id).to_dict()

benchmark_unique_docs = corpus_dict
benchmark_ground_truth = []

# 5. Map the questions to the correct papers
for _, row in qrels_df.iterrows():
    q_id = str(row['query_id'])
    d_id = str(row['doc_id'])
    
    # Only keep the mapping if the text actually exists in our downloaded tables
    if q_id in queries_dict and d_id in corpus_dict:
        benchmark_ground_truth.append({
            "query": str(queries_dict[q_id]),
            "correct_paper_id": str(d_id)
        })

print(f"✅ Successfully assembled {len(benchmark_ground_truth)} benchmark pairs. Ready for inference!")

Mapping the exact schemas found...
Loading queries...
Loading mappings...
Loading corpus...

✅ Extracted Tables: 380879 papers | 300 queries | 15000 mappings
Stitching them together for evaluation...
✅ Successfully assembled 0 benchmark pairs. Ready for inference!


In [7]:
# Convert our dictionary into flat lists for ChromaDB ingestion
benchmark_doc_ids = list(benchmark_unique_docs.keys())
benchmark_doc_texts = list(benchmark_unique_docs.values())

print(f"Ready to embed {len(benchmark_doc_texts)} unique documents!")

Ready to embed 380879 unique documents!


In [8]:
# ==========================================
# 3. THE MASTER EVALUATION LOOP
# ==========================================
MODELS_TO_TEST = [
    "all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-large-en-v1.5",
    "nomic-ai/nomic-embed-text-v1.5",
]

chroma_client = chromadb.Client()

for model_name in MODELS_TO_TEST:
    print("\n" + "=" * 50)
    print(f"🚀 TESTING MODEL: {model_name}")
    print("=" * 50)

    try:
        if "nomic" in model_name:
            emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name=model_name, 
                trust_remote_code=True,
                device="cuda" # <--- explicitly routes to Kaggle GPU
            )
        else:
            emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name=model_name,
                device="cuda" # <--- explicitly routes to Kaggle GPU
            )
        # -----------------------------------------
        # TEST 1: Synthetic Dataset
        # -----------------------------------------
        col_synth_name = f"synth_{model_name.replace('/', '-').replace('.', '-')}"
        try:
            chroma_client.delete_collection(name=col_synth_name)
        except Exception:
            pass

        col_synth = chroma_client.create_collection(
            name=col_synth_name,
            embedding_function=emb_fn,
            metadata={"hnsw:space": "cosine"},
        )
        col_synth.add(
            documents=[p["abstract"] for p in synthetic_docs],
            ids=[p["id"] for p in synthetic_docs],
        )

        synth_metrics = []
        for item in tqdm(
            synthetic_ground_truth, desc=f"[{model_name}] Synthetic Queries"
        ):
            start = time.perf_counter()
            res = col_synth.query(query_texts=[item["query"]], n_results=10)
            latency = (time.perf_counter() - start) * 1000
            synth_metrics.append(
                calculate_metrics(res["ids"][0], item["correct_paper_id"], latency)
            )

        print_scorecard(model_name, "Local Synthetic (100 papers)", synth_metrics)

        # -----------------------------------------
        # TEST 2: c1khoa Benchmark
        # -----------------------------------------
        # -----------------------------------------
        # TEST 2: c1khoa Benchmark (Optimized for Time)
        # -----------------------------------------
        col_bench_name = f"bench_{model_name.replace('/', '-').replace('.', '-')}"
        try:
            chroma_client.delete_collection(name=col_bench_name)
        except Exception:
            pass

        col_bench = chroma_client.create_collection(
            name=col_bench_name,
            embedding_function=emb_fn,
            metadata={"hnsw:space": "cosine"},
        )

        # SLICE 1: Only index the first 50,000 papers (The Haystack)
        # This prevents the 9-hour indexing wait.
        MAX_DOCS = 50000
        indexed_ids = benchmark_doc_ids[:MAX_DOCS]
        indexed_texts = benchmark_doc_texts[:MAX_DOCS]

        print(f"Indexing {MAX_DOCS} papers for {model_name}...")
        for i in range(0, len(indexed_ids), 500):
            col_bench.add(
                documents=indexed_texts[i : i + 500],
                ids=indexed_ids[i : i + 500],
            )

        # SLICE 2: Only run the first 100 queries (The Exam)
        # We must filter ground truth to only include papers that are actually in our 50k haystack
        available_ids = set(indexed_ids)
        limited_ground_truth = [
            item for item in benchmark_ground_truth 
            if item["correct_paper_id"] in available_ids
        ][:100]

        bench_metrics = []
        for item in tqdm(limited_ground_truth, desc=f"[{model_name}] c1khoa Queries (Top 100)"):
            start = time.perf_counter()
            res = col_bench.query(query_texts=[item["query"]], n_results=10)
            latency = (time.perf_counter() - start) * 1000
            bench_metrics.append(
                calculate_metrics(res["ids"][0], item["correct_paper_id"], latency)
            )

        print_scorecard(model_name, f"c1khoa Benchmark ({MAX_DOCS} docs, 100 queries)", bench_metrics)

    except Exception as e:
        print(f"❌ FAILED to evaluate {model_name}. Error: {e}")


🚀 TESTING MODEL: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

[nomic-ai/nomic-embed-text-v1.5] Synthetic Queries: 100%|██████████| 179/179 [00:03<00:00, 56.55it/s]



--------------------------------------------------
📂 DATASET: Local Synthetic (100 papers)
Recall@10:         0.9888 (98.9%)
MRR:               0.9844
nDCG@10:           0.9854
Avg Query Latency: 17.56 ms
--------------------------------------------------
Indexing 50000 papers for nomic-ai/nomic-embed-text-v1.5...


[nomic-ai/nomic-embed-text-v1.5] c1khoa Queries (Top 100): 0it [00:00, ?it/s]
